# 13 · Final §4.2 runs: can an improved sequence model use the history?

In notebook 09 the v2 cohorts carried information only in the order, trend and recency of
the coded history (the oracle's lead grew to 0.043), but our small sequence model did not
beat a bag-of-codes table. This tests the likely causes:

* **size**: hidden 64, 4 layers instead of hidden 16, 2 layers;
* **pooling**: `[CLS]` pooling instead of averaging over up to 256 mostly-noise tokens;
* **time**: `sequence_time` gives every code its recency `exp(t / 1 year)` through the
  numeric-value embedding, not only the `VISIT` markers.

Same three v2 cohorts, oracle and Cox [bag] as references, 5 splits.
Standalone: clones the code, needs Internet; GPU recommended.

## Options

In [ ]:
NAME = "13_final_sequential_ehr"

# The v2 EHR cohorts of notebook 09 (sat/data/simulate_ehr.py, design="v2"):
SOURCES = {"ehrsim2_level": "v2_level", "ehrsim2_mixed": "v2_mixed", "ehrsim2_order": "v2_order"}
MAX_TOKENS = 256
SEEDS = [0, 1, 2, 3, 4]

# Model variants. "small" = the settings of notebooks 06/09 (hidden 16, 2 layers, mean pooling).
# "large" = hidden 64, 4 layers, [CLS] pooling (token_emb=5) instead of averaging over up to
# 256 mostly-noise tokens, lr 1e-3.
LARGE = ["transformer_hidden_size=64", "transformer_num_hidden_layers=4", "transformer_num_attention_heads=4",
         "transformer_intermediate_size=128", "token_emb=5", "sentence_emb=1", "learning_rate=0.001"]
VARIANTS = [
    # (representation, model size)
    ("bag", "small"), ("bag", "large"),
    ("sequence", "small"), ("sequence", "large"),
    ("sequence_time", "large"),     # + recency exp(t / 1 year) on every code
]
RECIPE = "nllpch"

# --- execution --------------------------------------------------------------
SMOKE_TEST   = False    # False = the real run. True: 2 epochs, 1 seed, reduced plan, only to check
                        # that everything runs (~10-30 min); smoke results go to a separate folder.
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h)
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # code is cloned from here (nothing to upload)
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy (models, caches) after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        # prefer the code dataset over repo copies inside attached notebook outputs (stale code)
        cands = [f.parent.parent for f in Path("/kaggle/input").rglob("finetune.py")
                 if f.parent.name == "sat" and _is_repo(f.parent.parent)]
        cands.sort(key=lambda p: ("code" not in str(p).lower(), len(p.parts)))
        src = cands[0] if cands else None
        if src is not None:
            print("copying code from attached dataset:", src)
            shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(
                ".git", ".venv", "model-hub", "__pycache__", "results", "tb_logs"))
        elif GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            raise RuntimeError("Attach the dmmst code dataset (notebooks/new/README.md, step 1) or set GIT_URL.")
    RESULTS_ROOT = WORK / "results"
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "new"
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
print("repo:", REPO, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy is NEVER moved: pandas/pyarrow in the image are compiled against it.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            f"numpy=={_np.__version__}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
from scripts.runner import Run, Runner, dataset_overrides
from scripts.report import build as build_report

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input" -> your notebook) and the
# finished runs are copied back, so only the missing ones are trained.
if ON_KAGGLE:
    n_copied = 0
    for d in Path("/kaggle/input").rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
    print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Cohorts

In [ ]:
from scripts.runner import ehr_overrides
if SMOKE_TEST:
    SEEDS = SEEDS[:1]
    SOURCES = dict(list(SOURCES.items())[:1])
REPRS = sorted({rep for rep, _ in VARIANTS})
for src, scen in SOURCES.items():
    if not (REPO / "data" / src / "meta.json").is_file() or "sequence_time" not in json.loads(
            (REPO / "data" / src / "meta.json").read_text()).get("num_features", {}):
        cmd = [sys.executable, "-m", "sat.data.simulate_ehr", "--scenario", scen, "--out", f"data/{src}",
               "--max-tokens", str(MAX_TOKENS)] + (["--set", "n=1000"] if SMOKE_TEST else [])
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL)
    for rep in REPRS:
        r.prepare("ehrseq/survival", ehr_overrides(src, rep, REPO), label=f"{src}_{rep}")

def variant_overrides(src, rep, size):
    ov = ehr_overrides(src, rep, REPO)
    if size == "large":  # [CLS] takes one more position
        ov = [f"me_num_features={int(o.split('=')[1]) + 1}" if o.startswith("me_num_features=") else o
              for o in ov] + LARGE
    return ov

## Run

In [ ]:
runs = []
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    for src in SOURCES:
        base = {"dataset": src, "seed": s}
        runs.append(Run(f"{src}__oracle__s{s}", "oracle", "ehrseq/survival",
                        ehr_overrides(src, "sequence", REPO) + so, f"{src}_sequence", base | {"model": "oracle"}))
        runs.append(Run(f"{src}_bag__coxph__s{s}", "coxph", "ehrseq/coxph",
                        ehr_overrides(src, "bag", REPO) + so, f"{src}_bag", base | {"model": "coxph", "repr": "bag"}))
        for rep, size in VARIANTS:
            runs.append(Run(f"{src}_{rep}__ours_{RECIPE}_{size}__s{s}", "finetune", "ehrseq/survival",
                            variant_overrides(src, rep, size) + so + [f"tasks/losses={RECIPE}"], f"{src}_{rep}",
                            base | {"model": "ours", "recipe": RECIPE, "repr": rep, "variant_tag": size}))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Results
The question: does any sequence variant beat the bag table, and how much of the oracle's lead does it close? The paired table compares every variant with `bag (small)`.

In [ ]:
# ---- report + zip -----------------------------------------------------------------------
df = build_report([r.results_dir], r.results_dir / "report")
rep = r.results_dir / "report" / "REPORT.md"
if rep.is_file():
    display(Markdown(rep.read_text(encoding="utf-8")))
print("zip:", r.zip())

from scripts.runner import paired_vs
res = r.collect()
ours = res[res.get("model") == "ours"].copy()
ours["cfg"] = ours["repr"] + " (" + ours["variant_tag"] + ")"
for metric in ["ctd_weighted_avg", "true_order_brier", "brier_survtrace_weighted_avg"]:
    if metric in ours:
        t = paired_vs(ours, metric, by="cfg", baseline="bag (small)", pair_on="seed", within="dataset")
        t.to_csv(r.results_dir / "report" / f"variants_paired_{metric}.csv", index=False)
        display(t)

## Clean up
Keeps the Kaggle output small (results + zip only) and stops a later session from picking up this repo copy instead of the code dataset.

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)